# 00 — Setup & Authentication

**Scenario: the Cobalt Advisory Co-Pilot.** Over the next seven notebooks we build, layer by layer, an AI co-pilot for advisers at a fictional wealth-management firm. Every notebook reuses the same client persona — `Avery Chen`, age 47, moderate-aggressive, $1.42M AUM — so you can see the same task get progressively better service as we add capabilities from the Microsoft AI agents stack:

1. **Layer 1 — Models** (Chat Completions → Responses)
2. **Layer 2 — Foundry Agent Service** (hosted agents + managed tools)
3. **Layer 3 — Microsoft Agent Framework** (single & multi-agent workflows in code)
4. **Layer 4 — Microsoft Agent 365** (Entra Agent Identity, Work IQ MCP, Purview, observability)

This notebook does three things and nothing else:

- installs the base SDKs,
- verifies your Microsoft **Entra ID** login works against the Foundry project and Azure OpenAI resource,
- introduces the shared mock data you'll see in every subsequent notebook.

> **No API keys.** Every notebook in this series uses `DefaultAzureCredential`.

## Prerequisites

Before running this notebook:

1. An Azure subscription with an **Azure AI Foundry** project — yours: `Main-Project` on `r2d2-foundry-001`.
2. A model deployment named **`gpt-5.4`** (or whatever you put in `MODEL_DEPLOYMENT`) in the same Foundry resource.
3. Your identity has at minimum **`Cognitive Services OpenAI User`** and **`Azure AI Developer`** on the resource and project.
4. `az login` completed in this shell (`az account show` returns your account).
5. A `.env` file at the repo root — copy `.env.example` and edit if needed.
6. The repo's virtual env active: `source .venv/bin/activate`.

In [ ]:
# One-time install of the base SDKs used in notebooks #0–#3.
# MAF + Agent 365 packages are installed in the notebooks that need them.
%pip install --quiet --upgrade \
    "openai>=1.55" \
    "azure-identity>=1.19" \
    "python-dotenv>=1.0" \
    --pre "azure-ai-projects>=2.0.0b1"

## Step 1 — Load env & credentials

`load_env()` reads `.env` at the repo root, validates required vars, and returns a typed `Config`. The credential is `DefaultAzureCredential()` — works with `az login`, VS Code sign-in, managed identity, etc.

In [ ]:
import sys, pathlib

# Make the sibling `_common/` package importable when the notebook is run from notebooks/.
if str(pathlib.Path.cwd()) not in sys.path:
    sys.path.insert(0, str(pathlib.Path.cwd()))

from _common.env import load_env
from _common.clients import get_credential

cfg = load_env()
credential = get_credential()
print(cfg)

## Step 2 — Smoke-test the Azure OpenAI endpoint

We call **`gpt-5.4`** through the Chat Completions surface with `reasoning_effort="low"` and have it say hi as the Cobalt Co-Pilot. A successful response means:

- Entra token acquisition for scope `https://ai.azure.com/.default` works,
- your account has the right RBAC on the OpenAI resource,
- the deployment name in `.env` matches a real deployment.

In [ ]:
from _common.clients import get_openai_client

openai_client = get_openai_client(cfg, credential=credential)

response = openai_client.chat.completions.create(
    model=cfg.model_deployment,
    reasoning_effort=cfg.reasoning_effort,
    messages=[
        {"role": "system", "content": "You are the Cobalt Advisory Co-Pilot, a professional, concise assistant for wealth advisers."},
        {"role": "user",   "content": "In one sentence, introduce yourself to an adviser logging in for the first time."},
    ],
)

print("Model:   ", response.model)
print("Tokens:  ", response.usage.total_tokens if response.usage else "n/a")
print("Reply:   ", response.choices[0].message.content)

## Step 3 — Smoke-test the Foundry project endpoint

Same identity, different surface: this connects to your Foundry **project** (not the bare OpenAI resource). Listing connections is a harmless read that proves project-level RBAC works — we'll use the project client heavily in notebooks #3 and #6.

In [ ]:
from _common.clients import get_project_client

project = get_project_client(cfg, credential=credential)

print("Project endpoint:", cfg.project_endpoint)
print("Connections in the project:")
for conn in project.connections.list():
    print(f"  - {conn.name:30s}  type={conn.type}")

## Step 4 — Meet the scenario data

Every notebook in the series imports `CLIENT_PROFILE`, `HOLDINGS`, and the `portfolio_summary()` helper from `_common.scenario`. Skim the output below — this is the only client you need to remember.

In [ ]:
from pprint import pprint
from _common.scenario import CLIENT_PROFILE, HOLDINGS, WATCHLIST, portfolio_summary

print("=== Client ===")
pprint(CLIENT_PROFILE)

print("\n=== Holdings ===")
for h in HOLDINGS:
    print(f"  {h.symbol:5s} {h.asset_class:12s} mv=${h.market_value_usd:>11,.2f}  {h.name}")

print("\n=== Watchlist ===")
print(" ", ", ".join(WATCHLIST))

print("\n=== Portfolio summary ===")
pprint(portfolio_summary())

## You're set — what's next

If both smoke tests above returned content, your environment is wired correctly. The remaining notebooks each focus on one layer of the stack:

| # | Notebook | What you'll build |
|---|---|---|
| 1 | `01_chat_completions.ipynb` | Simple one-shot adviser Q&A with `gpt-5.4`. |
| 2 | `02_responses_api.ipynb` | Multi-turn portfolio Q&A using the **Responses API** with `web_search` + PDF input. |
| 3 | `03_foundry_agent_service.ipynb` | A hosted **Portfolio Review Agent** with `file_search` over research PDFs. |
| 4 | `04_maf_single_agent.ipynb` | Re-author the same agent in code with **Microsoft Agent Framework**. |
| 5 | `05_maf_multi_agent_workflow.ipynb` | Sequential analyst → tax-optimizer → compliance → brief-writer workflow. |
| 6 | `06_maf_deploy_as_hosted_agent.ipynb` | Deploy the MAF workflow as a Foundry Hosted Agent. |
| 7 | `07_agent_365_governance.ipynb` | Wrap the agent with **Microsoft Agent 365** — Entra Agent Identity, Work IQ MCP, Purview, OTel. The cherry on top. |

Each is independently runnable as long as notebook #0 worked.